# Titans Board v2.0 — Colab クイックスタート

CFO / CLO / CEO / 監査役 の AI 取締役会システム。

---

## ▶ Path A — Anthropic API（推奨・GPU不要・モデルDL不要）

**Step A1〜A4 だけで動く。** Ollamaもモデルダウンロードも不要。セッションが落ちても A2 から再実行するだけ。

必要なもの: `ANTHROPIC_API_KEY`（[console.anthropic.com](https://console.anthropic.com) で取得）

---
## ▶ Path B — Ollama（無料・GPUランタイム推奨）

API キー不要。ただしセッションごとに Ollama + モデル（2.4GB）を再DL。  
Step B1〜B3 の後、A2 から合流する。

---
## Step A1 — リポジトリ取得 & パッケージインストール

**セッションが落ちても、ランタイムが生きていればこのセルはスキップ可。**  
`/content/titans-board` が存在すれば clone はスキップし、最新コードに pull するだけ。

In [ ]:
import os, subprocess

REPO = "/content/titans-board"
BRANCH = "claude/titans-board-v2-design-pb81x7"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH,
                    "https://github.com/nori1234/Stock_Tracker-657.git", REPO], check=True)
    print("Clone 完了")
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
    print("Pull 完了")

# 実行中のコミットを表示（バグ報告時に貼ってください）
r = subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True)
print("Commit:", r.stdout.strip())

%cd {REPO}

In [ ]:
# パッケージインストール（crewai==1.14.6 を強制）
!pip install -q -r requirements.txt

import crewai
print("crewai:", crewai.__version__)
assert crewai.__version__ == "1.14.6", (
    "バージョン不一致 — ランタイムを再起動してこのセルを再実行してください")

---
## Step A2 — API キーを設定

Colab 左メニューの 🔑「シークレット」に `ANTHROPIC_API_KEY` を登録してから実行。  
（登録すればセッションをまたいでも毎回入力不要）

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("API キー設定: OK（先頭8文字:", os.environ["ANTHROPIC_API_KEY"][:8], "...)")

# 接続確認
!python main.py --anthropic --health-check

`Status: OK` が出れば準備完了。`Status: FAIL` の場合はキーが正しいか確認してください。

---
## Step A3 — 知識ベース & 長期記憶を設定（任意）

In [ ]:
# サンプル知識ファイルを取り込む
!python main.py --anthropic --ingest ./knowledge

In [ ]:
# 禁止事項・経営方針を長期記憶に登録（会議ごとに自動注入される）
!python main.py --anthropic --remember "ギャンブル・アダルト関連事業への参入禁止" --category 禁止事項
!python main.py --anthropic --remember "3年以内の黒字化を全事業の必須条件とする" --category 経営方針
!python main.py --anthropic --memories

---
## Step A4 — 取締役会を開催する

CFO → CLO → CEO草稿 → 監査役 → CEO最終 の順で審議が進みます。  
Haiku で **1〜2分**程度。`--model claude-sonnet-4-6` にするとより高品質（約30〜50円/回）。

| モデル | 速度 | コスト目安/1会議 |
|--------|------|------------------|
| `claude-haiku-4-5-20251001`（デフォルト） | 速い | 約2〜5円 |
| `claude-sonnet-4-6` | 普通 | 約30〜50円 |

In [ ]:
AGENDA = "新規事業として、AIを活用した医療診断支援サービスを日本市場で展開したい。初期投資5億円、3年でのROI達成が目標。取締役会の判断を仰ぎたい。"

# Haiku（速い・安い）
!python main.py --anthropic "{AGENDA}"

# Sonnet（より高品質）を使う場合は上をコメントアウトして↓を使う
# !python main.py --anthropic --model claude-sonnet-4-6 "{AGENDA}"

In [ ]:
# 保存されたレポートを確認
import glob, json, pathlib
files = sorted(glob.glob("outputs/meeting_*.json"))
if files:
    latest = json.loads(pathlib.Path(files[-1]).read_text())
    print(f"保存先: {files[-1]}")
    print("\n=== CEO最終判断 ===")
    print(latest.get("ceo_final", "")[:1500])
else:
    print("出力ファイルがありません")

---
---
## Path B — Ollama（API キー不要・無料）

ランタイム: T4 GPU 推奨（メニュー → ランタイム → ランタイムのタイプを変更）  
**注意: セッションごとに Ollama + モデル（約2.4GB）を再ダウンロードします。**  
B1〜B3 が完了したら Step A2 に戻って接続確認を行い、A4 の `--anthropic` を外して実行。

In [ ]:
%%bash
set -e
# zstd: Ollama アーカイブ展開に必須（Colab 未導入のため先にインストール）
command -v zstd > /dev/null || apt-get install -y -q zstd
echo "--- zstd: $(zstd --version)"
curl -fsSL https://ollama.com/install.sh | sh
echo "--- ollama: $(ollama --version)"
echo "=== Step B1 完了 ==="

In [ ]:
import shutil, subprocess, time, urllib.request

assert shutil.which("ollama"), "ollama が見つかりません。Step B1 を再実行してください"

proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        print("Ollama サーバー: OK")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama サーバーが起動しませんでした")

In [ ]:
# モデル取得（約2.4GB・数分かかります）
!ollama pull qwen3:4b
print("=== Step B3 完了 — Step A2 に進んでください（--anthropic フラグは不要）===")

---
## トラブルシュート

| 症状 | 対処 |
|------|------|
| `Status: FAIL`（Anthropic） | ANTHROPIC_API_KEY が正しいか確認 |
| `requires zstd`（Ollama） | Step B1 の `apt-get install zstd` が含まれているか確認 |
| `crewai` バージョン不一致 | ランタイムを再起動 → Step A1 から再実行 |
| セッション切れ後（Anthropic） | Step A2 から再実行するだけ（clone/install不要の場合が多い） |
| セッション切れ後（Ollama） | Step B1〜B3 を再実行（Ollamaと2.4GBモデルが消える） |